## Preamble

In [ ]:
# Import libraries
import pandas as pd
import plotly.express as px
import plotly.io as pio
import sqlite3

In [ ]:
# Directory
import os
os.chdir('/Users/tasia/Desktop/Data Analysis/Assessment')

In [ ]:
# Load data from database
conn = sqlite3.connect("wdi_cross_sectional_gdp_unemployment_2023.db")

df_gdp = pd.read_sql_query("SELECT * FROM gdp", conn)
df_unp = pd.read_sql_query("SELECT * FROM unemployment", conn)

conn.close()

# Eyeball
print(df_gdp.head())
print(df_unp.head())

In [ ]:
# Convert columns to numeric and replace invalid values with NaN
df_gdp['2023 [YR2023]'] = pd.to_numeric(df_gdp['2023 [YR2023]'], errors='coerce')
df_unp['2023 [YR2023]'] = pd.to_numeric(df_unp['2023 [YR2023]'], errors='coerce')

In [ ]:
# Merge dataframes (inner join!)
df = pd.merge(left = df_gdp, 
              right = df_unp,
              how = 'inner',
              left_on = ['Country Code'],
              right_on = ['Country Code'],
              )
# Rename columns
df = df.rename(columns={
    '2023 [YR2023]_x': 'GDP per capita (2023)',
    '2023 [YR2023]_y': 'Unemployment rate (2023)'})

# Eyeball
df.head()

In [ ]:
# Adding region information
import requests

url = "https://api.worldbank.org/v2/country?format=json&per_page=400"

response = requests.get(url)
response.raise_for_status()
countries = response.json()[1]

# Dataframe with country codes and regions
regions = pd.DataFrame([{"Country Code": country["id"],
                         "Region": country["region"]["value"]}
                        for country in countries])

# Merging region information with GDP and unemployment data
df = pd.merge(df, regions, on="Country Code", how="left")

# Removing missing values and World Bank aggregates
df_regions = df.dropna(subset=["GDP per capita (2023)","Unemployment rate (2023)","Region"])

df_regions = df_regions[df_regions["Region"] != "Aggregates"]

# List available regions
regions_list = sorted(df_regions["Region"].unique())

# Eyeball
df[["Country Name_x", "Region"]].head()

In [ ]:
# Display summary statistics (Table 1)
df.describe()

### Interpretation and Discussing the Output

## Plotting 6 Interactive Graphs

Some graphs and plots include a non-interactive graph/plot versions to show work process

## Bar Charts

In [ ]:
# Plotting a GDP per capita Bar Chart for EU15 Countries in 2023 (Non-interactive)
fig = px.bar(df[(df["Country Name_x"].isin(["Austria",
                                     "Belgium",
                                     "Denmark",
                                     "Finland",
                                     "France",
                                     "Germany",
                                     "Greece",
                                     "Ireland",
                                     "Italy",
                                     "Luxembourg",
                                     "Netherlands",
                                     "Portugal",
                                     "Spain",
                                     "Sweden",
                                     "United Kingdom"]))], 
              x = "Country Name_x", 
              y = "GDP per capita (2023)",
              title = "GDP per capita Across EU15 Countries in 2023")
fig.show()

In [ ]:
# Plotting an Interactive GDP per capita Bar Chart for EU15 Countries in 2023 by European Region (Figure 1)
# Selecting the EU15 countries
eu15 = ["Austria", "Belgium", "Denmark", "Finland", "France", "Germany", "Greece", "Ireland", "Italy", "Luxembourg", "Netherlands", 
        "Portugal", "Spain", "Sweden", "United Kingdom"]

df_eu15 = df[df["Country Name_x"].isin(eu15)].copy()

# Group countries into European regions
regions = {"Northern Europe": ["Denmark", "Finland", "Ireland", "Sweden", "United Kingdom"],
           "Western Europe": ["Austria", "Belgium", "France", "Germany", "Luxembourg", "Netherlands"],
           "Southern Europe": ["Greece", "Italy", "Portugal", "Spain"]}

# Creating a region column
for region, countries in regions.items():
    df_eu15.loc[
        df_eu15["Country Name_x"].isin(countries), "European Region"
    ] = region

# Creating the bar chart
fig = px.bar(
    df_eu15,
    x="Country Name_x",
    y="GDP per capita (2023)",
    title="GDP per capita Across EU15 Countries by European Region in 2023")

# Creating the dropdown menu
buttons = [dict(label="All EU15 countries",method="update",
                args=[{"x": [df_eu15["Country Name_x"].tolist()],
                       "y": [df_eu15["GDP per capita (2023)"].tolist()]}])]

for region in regions:
    region_data = df_eu15[df_eu15["European Region"] == region]

    buttons.append(dict(label=region,method="update",
                        args=[{"x": [region_data["Country Name_x"].tolist()],
                               "y": [region_data["GDP per capita (2023)"].tolist()]}]))

# Adding the dropdown menu above the chart
fig.update_layout(updatemenus=[dict(buttons=buttons,
                                    direction="down",
                                    showactive=True,
                                    x=0,
                                    y=1.35,
                                    xanchor="left",
                                    yanchor="top")],
                  margin=dict(t=180),
                  title=dict(x=0.5,xanchor="center",y=0.90))

fig.show()

### Interpretation and Discussing the Output

In [ ]:
# Plotting an Unemployment Rate Bar Chart for EU15 Countries in 2023 (Non-interactive)
fig = px.bar(df[(df["Country Name_y"].isin(["Austria",
                                     "Belgium",
                                     "Denmark",
                                     "Finland",
                                     "France",
                                     "Germany",
                                     "Greece",
                                     "Ireland",
                                     "Italy",
                                     "Luxembourg",
                                     "Netherlands",
                                     "Portugal",
                                     "Spain",
                                     "Sweden",
                                     "United Kingdom"]))], 
              x = "Country Name_y", 
              y = "Unemployment rate (2023)",
              title = "Unemployment Rate Across EU15 Countries in 2023")
fig.show()

In [ ]:
# Plotting an Interactive Unemployment Rate Bar Chart for EU15 Countries in 2023 by European Region (Figure 2)
# Selecting EU15 countries
eu15 = ["Austria", "Belgium", "Denmark", "Finland", "France", "Germany", "Greece", "Ireland", "Italy", "Luxembourg", "Netherlands", 
        "Portugal", "Spain", "Sweden", "United Kingdom"]

df_eu15 = df[df["Country Name_y"].isin(eu15)].copy()

# Group countries into European regions
regions = {"Northern Europe": ["Denmark", "Finland", "Ireland", "Sweden", "United Kingdom"],
           "Western Europe": ["Austria", "Belgium", "France", "Germany", "Luxembourg", "Netherlands"],
           "Southern Europe": ["Greece", "Italy", "Portugal", "Spain"]}

# Creating a region column
for region, countries in regions.items():
    df_eu15.loc[df_eu15["Country Name_y"].isin(countries), "European Region"] = region

# Creating the bar chart
fig = px.bar(df_eu15,
    x="Country Name_y",
    y="Unemployment rate (2023)",
    title="Unemployment Rates Across EU15 Countries by European Region in 2023")

# Creating dropdown menu
buttons = [dict(
        label="All EU15 countries",
        method="update",
        args=[{"x": [df_eu15["Country Name_y"].tolist()],
               "y": [df_eu15["Unemployment rate (2023)"].tolist()]}])]

for region in regions:
    region_data = df_eu15[df_eu15["European Region"] == region]

    buttons.append(dict(label=region,
            method="update",
            args=[{"x": [region_data["Country Name_y"].tolist()],
                   "y": [region_data["Unemployment rate (2023)"].tolist()]}]))

# Adding dropdown menu above the chart
fig.update_layout(updatemenus=[dict(buttons=buttons,
                                    direction="down",
                                    showactive=True,
                                    x=0,
                                    y=1.35,
                                    xanchor="left",
                                    yanchor="top")],
                  margin=dict(t=180),
                  title=dict(
                      x=0.5,
                      xanchor="center",
                      y=0.90),yaxis_title="Unemployment rate (%)")
fig.show()

### Interpretation and Discussing the Output

In [ ]:
# Plotting a Clustered Bar Chart of All Countries by World Bank Region in 2023 (Non-interactive)

# Calculating average GDP per capita
average_gdp = df_regions["GDP per capita (2023)"].mean()

# Grouping countries by GDP per capita
df_clustered = df_regions.copy()

df_clustered["GDP Group"] = "Below-average GDP"
df_clustered.loc[df_clustered["GDP per capita (2023)"] >= average_gdp, "GDP Group"] = "Above-average GDP"

# Calculating average unemployment by region and GDP group
df_clustered = df_clustered.groupby(["Region", "GDP Group"], as_index = False)["Unemployment rate (2023)"].mean()

# Plotting clustered bar chart
fig = px.bar(df_clustered,
             x = "Region",
             y = "Unemployment rate (2023)",
             color = "GDP Group",
             barmode = "group",
             title = "Average Unemployment by World Bank Region and GDP group in 2023")

fig.update_layout(yaxis_title = "Average unemployment rate (%)")

fig.show()

In [ ]:
# Plotting an Interactive Clustered Bar Chart of All Countries by World Bank Region in 2023 (Figure 3)
fig = px.bar(df_clustered,
             x = "Region",
             y = "Unemployment rate (2023)",
             color = "GDP Group",
             barmode = "group",
             title = "Average Unemployment by World Bank Region and GDP group in 2023")

# Creating dropdown buttons
regions_list = sorted(df_clustered["Region"].unique())

buttons = [dict(label = "All regions",
                method = "update",
                args = [{"x": [df_clustered[df_clustered["GDP Group"] == trace.name]["Region"].tolist() for trace in fig.data],
                         "y": [df_clustered[df_clustered["GDP Group"] == trace.name]["Unemployment rate (2023)"].tolist() for trace in fig.data]}])]

for region in regions_list:
    region_data = df_clustered[df_clustered["Region"] == region]

    buttons.append(dict(label = region,
                        method = "update",
                        args = [{"x": [region_data[region_data["GDP Group"] == trace.name]["Region"].tolist() for trace in fig.data],
                                 "y": [region_data[region_data["GDP Group"] == trace.name]["Unemployment rate (2023)"].tolist() for trace in fig.data]}]))

# Adding dropdown menu
fig.update_layout(updatemenus = [dict(buttons = buttons,
                        direction = "down",
                        showactive = True,
                        x = 0,
                        y = 1.35,
                        xanchor = "left",
                        yanchor = "top")],
    margin = dict(t = 180),
    title = dict(x = 0.5,
                 xanchor = "center",
                 y = 0.90),
    yaxis_title = "Average unemployment rate (%)")

fig.show()

### Interpretation and Discussing the Output

## Scatter Plots

In [ ]:
# Plotting a Scatter Plot of GDP per capita and Unemployment in 2023 (Non-interactive)
fig = px.scatter(df, 
                 x = "GDP per capita (2023)", 
                 y = "Unemployment rate (2023)",
                 hover_data = ["Country Name_x"],
                 title = "GDP per capita and Unemployment Rates in 2023")
fig.show()

In [ ]:
# Plotting an Interactive Scatter Plot of GDP per capita and Unemployment in 2023 by Region (Figure 4)
# Creating the interactive scatter plot with region selection
fig = px.scatter(
    df_regions,
    x="GDP per capita (2023)",
    y="Unemployment rate (2023)",
    color="Region",
    hover_name="Country Name_x",
    title="GDP per capita and Unemployment by Region in 2023")

# Creating dropdown buttons for each region
regions_list = sorted(df_regions["Region"].unique())

buttons = [dict(
        label="All regions",
        method="update",
        args=[{"visible": [True] * len(fig.data)}])]

for region in regions_list:
    buttons.append(dict(label=region,
            method="update",
            args=[{"visible": [trace.name == region for trace in fig.data]}]))


# Adding dropdown menu above the scatter plot
fig.update_layout(updatemenus=[dict(
    buttons=buttons,
    direction="down",
    showactive=True,
    x=0,
    y=1.35,
    xanchor="left",
    yanchor="top")],
                  margin=dict(t=180),
                  title=dict(
                      x=0.5,
                      xanchor="center",
                      y=0.90))
fig.show()

### Interpretation and Discussing the Output

## Histograms

In [ ]:
# Plotting a Bivariate Histogram (Heatmap) of GDP per capita and Unemployment in 2023 (Non-interactive)
fig = px.density_heatmap(df, 
                         x = "GDP per capita (2023)", 
                         y = "Unemployment rate (2023)",
                         nbinsx = 30, 
                         nbinsy = 30)
fig.show()

In [ ]:
# Plotting an Interactive Bivariate Histogram (Heatmap) of GDP per capita and Unemployment in 2023 by Region (Figure 5)
# Create a heatmap for all regions
fig = px.density_heatmap(df_regions,
                         x="GDP per capita (2023)",
                         y="Unemployment rate (2023)",
                         nbinsx=30,
                         nbinsy=30,
                         title="GDP per capita and Unemployment by Region in 2023")

# Creating dropdown buttons
buttons = [dict(label="All regions",method="update",args=[{"x": [df_regions["GDP per capita (2023)"].tolist()],
                                                           "y": [df_regions["Unemployment rate (2023)"].tolist()]}])]

for region in regions_list:
    region_data = df_regions[df_regions["Region"] == region]

    buttons.append(dict(label=region,method="update",args=[{"x": [region_data["GDP per capita (2023)"].tolist()],
                                                            "y": [region_data["Unemployment rate (2023)"].tolist()]}]))
    
# Adding dropdown menu above the heatmap
fig.update_layout(updatemenus=[dict(buttons=buttons,
                                    direction="down",
                                    showactive=True,
                                    x=0,
                                    y=1.35,
                                    xanchor="left",
                                    yanchor="top")],
                  margin=dict(t=180),
                  title=dict(
                      x=0.5,
                      xanchor="center",
                      y=0.90))

fig.show()

### Interpretation and Discussing the Output

In [ ]:
# Plotting a Heatmap with Marginal Histograms of GDP per capita and Unemployment in 2023 (Non-interactive)
fig = px.density_heatmap(df, 
                         x = "GDP per capita (2023)", 
                         y = "Unemployment rate (2023)",
                         marginal_x = "histogram", 
                         marginal_y = "histogram",
                         nbinsx = 30, 
                         nbinsy = 30)
fig.show()

In [ ]:
# Plotting an Interactive Heatmap with Marginal Histograms of GDP per capita and Unemployment in 2023 by Region (Figure 6)
# Using the same valid country observations for every selection
df_heatmap = df_regions.copy()

# Creating the interactive heatmap
fig = px.density_heatmap(df_heatmap,
                         x="GDP per capita (2023)",
                         y="Unemployment rate (2023)",
                         marginal_x="histogram",
                         marginal_y="histogram",
                         nbinsx=30,
                         nbinsy=30,
                         title="GDP per capita and Unemployment with Marginal Histograms by Region in 2023")

# Creating dropdown options
regions_list = sorted(df_heatmap["Region"].dropna().unique())

# Heatmap, top histogram and right histogram

def make_region_update(data):
    gdp = data["GDP per capita (2023)"].tolist()
    unemployment = data["Unemployment rate (2023)"].tolist()

    return {"x": [gdp, gdp, None], "y": [unemployment, None, unemployment]}

buttons = [dict(label="All regions", method="update", args=[make_region_update(df_heatmap)])]

for region in regions_list:
    region_data = df_heatmap[df_heatmap["Region"] == region]
    
    buttons.append(dict(label=region,
                        method="update",
                        args=[make_region_update(region_data)]))

# Adding dropdown above the chart
fig.update_layout(updatemenus=[dict(buttons=buttons,
                                    direction="down",
                                    showactive=True,
                                    x=0,
                                    y=1.35,
                                    xanchor="left",
                                    yanchor="top")],
                  margin=dict(t=180),
                  title=dict(x=0.5,
                      xanchor="center",
                             y=0.90))

fig.show()

### Interpretation and Discussing the Output

## Bubble Chart

In [ ]:
# Downloading Population Data for 2023
import requests

url = "https://api.worldbank.org/v2/country/all/indicator/SP.POP.TOTL?format=json&date=2023&per_page=500"
data = requests.get(url).json()[1]

# Creating dataframe
df_population = pd.DataFrame(data)
df_population["Country Code"] = df_population["countryiso3code"]
df_population["Population (2023)"] = df_population["value"]

# Merge dataframes (inner join!)
df_bubble = pd.merge(left = df_regions,
                     right = df_population[["Country Code", "Population (2023)"]],
                     how = "inner",
                     on = "Country Code")

# Remove missing population values
df_bubble = df_bubble.dropna(subset = ["Population (2023)"])

# Eyeball
df_bubble.head()

In [ ]:
# Plotting an Interactive Bubble Chart of GDP per capita and Unemployment in 2023 by Region (Figure 7)
fig = px.scatter(df_bubble,
                 x = "GDP per capita (2023)",
                 y = "Unemployment rate (2023)",
                 size = "Population (2023)",
                 color = "Region",
                 hover_name = "Country Name_x",
                 size_max = 55,
                 title = "GDP per capita, Unemployment and Population by Region in 2023")

# Creating dropdown buttons for each region
regions_list = sorted(df_bubble["Region"].unique())

buttons = [dict(label = "All regions",
                method = "update",
                args = [{"visible": [True] * len(fig.data)}])]

for region in regions_list:
    buttons.append(dict(label = region,
                        method = "update",
                        args = [{"visible": [trace.name == region for trace in fig.data]}]))

# Adding dropdown menu
fig.update_layout(updatemenus = [dict(buttons = buttons,
                        direction = "down",
                        showactive = True,
                        x = 0,
                        y = 1.35,
                        xanchor = "left",
                        yanchor = "top")],
    margin = dict(t = 180),
    title = dict(x = 0.5,
                 xanchor = "center",
                 y = 0.90))

fig.show()

### Interpretation and Discussing the Output